# Audi Q4 e-tron — Time-to-Sell Prediction

**Goal:** Predict how quickly a listing disappears from the market using two complementary approaches:

**Part A — Binary Classification:** Predict whether a listing will be sold/removed within **14 days** (`sold_within_14d`).  
- Dataset: 1,954 eligible listings, **perfectly balanced 50/50 target**
- Models compared: Logistic Regression, Random Forest, XGBoost, LightGBM
- Evaluation: ROC-AUC, F1, PR-AUC, Brier Score
- Top 2 tuned with Optuna (100 trials each)

**Part B — Survival Analysis:** Model the full time-to-event distribution using `duration_days` and `event_sold_or_removed` (censored data).  
- Models: Cox Proportional Hazards, Weibull AFT
- Evaluation: C-index (concordance)
- Kaplan-Meier curves by model variant, price band, seller type

**Run order:** Run `02_price_prediction_model.ipynb` first (not required here but needed for `03_deal_finder.ipynb`).

In [ ]:
# ── 1. Dependencies ────────────────────────────────────────────────────────────
import subprocess, sys
for pkg in ['scikit-learn', 'xgboost', 'lightgbm', 'optuna', 'shap', 'lifelines']:
    try:
        __import__(pkg.replace('-', '_').split('[')[0])
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])

import warnings, json
from pathlib import Path
from datetime import datetime

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import StratifiedKFold, cross_val_score, cross_validate, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OrdinalEncoder, OneHotEncoder, label_binarize
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    roc_auc_score, f1_score, brier_score_loss, average_precision_score,
    roc_curve, precision_recall_curve, make_scorer, classification_report
)

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

import optuna
from optuna.samplers import TPESampler
optuna.logging.set_verbosity(optuna.logging.WARNING)

import shap

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', context='notebook')
plt.rcParams['figure.figsize'] = (12, 5)

RANDOM_STATE = 42
MODELS_DIR   = Path('saved_models')
MODELS_DIR.mkdir(exist_ok=True)

SALE_DATA_PATH     = Path('../Preprocessing/outputs/audi_q4_sale_14d_model_dataset.parquet')
SURVIVAL_DATA_PATH = Path('../Preprocessing/outputs/audi_q4_survival_model_dataset.parquet')

print('All imports OK.')
print(f'Sale-14d data exists: {SALE_DATA_PATH.exists()}')
print(f'Survival data exists: {SURVIVAL_DATA_PATH.exists()}')

---
# Part A — Binary Classification: Sold Within 14 Days

In [ ]:
# ── Load data & define features ────────────────────────────────────────────────
df = pd.read_parquet(SALE_DATA_PATH)
TARGET = 'sold_within_14d'

print(f'Shape: {df.shape}')
print(f'\nTarget balance:')
vc = df[TARGET].value_counts(dropna=False)
print(vc.to_frame('count').assign(pct=vc / len(df) * 100))

# Features — same as price model but NOW we INCLUDE price_eur (available at decision time)
# and price_per_kw/hp/range (they're not leakage for sale-speed prediction)
NUMERIC_FEATURES = [
    'price_eur',
    'mileage_km', 'vehicle_age_months', 'power_kw', 'power_hp',
    'electric_range_km', 'wltp_consumption_kwh_100km',
    'seller_rating_stars', 'seller_rating_count',
    'image_count', 'previous_owner_count', 'door_count', 'seat_count',
    'warranty_months', 'mileage_per_month',
    'price_per_kw', 'price_per_hp', 'price_per_range_km',
]

CATEGORICAL_ORDINAL = ['model_number_v2']
CATEGORICAL_NOMINAL = [
    'variant', 'seller_type', 'body_type', 'paint_type',
    'exterior_color', 'interior_color', 'upholstery_material',
]

BINARY_FEATURES = [
    'is_conditional_price_clean', 'available_now_clean',
    'warranty_exists_clean', 'has_full_service_history_clean', 'had_accident_clean',
    'is_sportback', 'is_quattro', 'is_s_line',
    'has_matrix', 'has_pano', 'has_ahk', 'has_hud', 'has_acc', 'has_camera',
    'duplicate_listing_id', 'seller_has_rating',
    'has_warranty_months', 'has_battery_info', 'has_city_range', 'has_charging_time',
]

# Filter to columns that exist
NUMERIC_FEATURES    = [c for c in NUMERIC_FEATURES    if c in df.columns]
CATEGORICAL_NOMINAL = [c for c in CATEGORICAL_NOMINAL if c in df.columns]
BINARY_FEATURES     = [c for c in BINARY_FEATURES     if c in df.columns]

ALL_FEATURES = NUMERIC_FEATURES + CATEGORICAL_ORDINAL + CATEGORICAL_NOMINAL + BINARY_FEATURES

X = df[ALL_FEATURES].copy()
y = df[TARGET].astype(int).copy()

print(f'\nFeatures: {len(ALL_FEATURES)} | Samples: {len(X)} | Missing target: {y.isna().sum()}')

In [ ]:
# ── Target & feature distributions ────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Class balance
axes[0].bar(['Not sold\n(14d)', 'Sold\n(14d)'], [y.eq(0).sum(), y.eq(1).sum()],
            color=['#F28E2B', '#4C78A8'])
axes[0].set_title('Class Balance — sold_within_14d')
axes[0].set_ylabel('Count')
for i, v in enumerate([y.eq(0).sum(), y.eq(1).sum()]):
    axes[0].text(i, v + 10, f'{v} ({v/len(y)*100:.1f}%)', ha='center', fontsize=10)

# Price vs sold_within_14d
if 'price_eur' in df.columns:
    sns.boxplot(data=df, x=TARGET, y='price_eur', ax=axes[1], palette=['#F28E2B', '#4C78A8'])
    axes[1].set_title('Price vs 14-Day Sale'); axes[1].set_ylabel('Price (€)')

# Mileage vs sold_within_14d
if 'mileage_km' in df.columns:
    sns.boxplot(data=df, x=TARGET, y='mileage_km', ax=axes[2], palette=['#F28E2B', '#4C78A8'])
    axes[2].set_title('Mileage vs 14-Day Sale'); axes[2].set_ylabel('Mileage (km)')

plt.suptitle('Target Distribution & Key Feature Associations', fontsize=12)
plt.tight_layout()
plt.show()

# Sale rate by model variant
if 'model_number_v2' in df.columns:
    sale_rate = df.groupby('model_number_v2')[TARGET].mean().sort_index()
    print('\n14-day sale rate by model variant:')
    print(sale_rate.map(lambda x: f'{x:.1%}'))

In [ ]:
# ── Preprocessing pipeline ─────────────────────────────────────────────────────
model_number_order = [['35', '40', '45', '50', '55']]

def make_preprocessor():
    return ColumnTransformer([
        ('num', Pipeline([
            ('i', SimpleImputer(strategy='median')),
            ('s', StandardScaler()),
        ]), NUMERIC_FEATURES),
        ('ord', Pipeline([
            ('i', SimpleImputer(strategy='constant', fill_value='missing')),
            ('e', OrdinalEncoder(
                categories=model_number_order,
                handle_unknown='use_encoded_value', unknown_value=-1
            )),
        ]), CATEGORICAL_ORDINAL),
        ('nom', Pipeline([
            ('i', SimpleImputer(strategy='constant', fill_value='missing')),
            ('e', OneHotEncoder(handle_unknown='ignore', sparse_output=False, min_frequency=5)),
        ]), CATEGORICAL_NOMINAL),
        ('bin', Pipeline([
            ('i', SimpleImputer(strategy='constant', fill_value=0)),
        ]), BINARY_FEATURES),
    ], remainder='drop')

# Verify pipeline
test_pre = make_preprocessor()
test_out = test_pre.fit_transform(X.head(5), y.head(5))
print(f'Preprocessor OK — output shape for 5 rows: {test_out.shape}')

In [ ]:
# ── Baseline Model Comparison (5-fold Stratified CV) ──────────────────────────
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

scoring = {
    'roc_auc': 'roc_auc',
    'f1':      make_scorer(f1_score),
    'pr_auc':  make_scorer(average_precision_score, needs_proba=True),
    'brier':   make_scorer(brier_score_loss, needs_proba=True, greater_is_better=False),
}

clf_models = {
    'Logistic Regression': Pipeline([
        ('pre', make_preprocessor()),
        ('clf', LogisticRegression(C=0.1, max_iter=500, random_state=RANDOM_STATE, n_jobs=-1))
    ]),
    'Random Forest': Pipeline([
        ('pre', make_preprocessor()),
        ('clf', RandomForestClassifier(
            n_estimators=300, n_jobs=-1, random_state=RANDOM_STATE
        ))
    ]),
    'XGBoost': Pipeline([
        ('pre', make_preprocessor()),
        ('clf', XGBClassifier(
            n_estimators=500, learning_rate=0.05, max_depth=5,
            subsample=0.8, colsample_bytree=0.8,
            random_state=RANDOM_STATE, verbosity=0, n_jobs=-1,
            eval_metric='logloss', use_label_encoder=False
        ))
    ]),
    'LightGBM': Pipeline([
        ('pre', make_preprocessor()),
        ('clf', LGBMClassifier(
            n_estimators=500, learning_rate=0.05, max_depth=5,
            num_leaves=31, subsample=0.8, colsample_bytree=0.8,
            random_state=RANDOM_STATE, verbosity=-1, n_jobs=-1
        ))
    ]),
}

print('Running 5-fold stratified CV for 4 models...')
clf_results = {}

for name, model in clf_models.items():
    cv_res = cross_validate(model, X, y, cv=cv, scoring=scoring, n_jobs=-1)
    clf_results[name] = {
        'ROC-AUC':  cv_res['test_roc_auc'].mean(),
        'F1':       cv_res['test_f1'].mean(),
        'PR-AUC':   cv_res['test_pr_auc'].mean(),
        'Brier':    -cv_res['test_brier'].mean(),
        'AUC_std':  cv_res['test_roc_auc'].std(),
    }
    print(f'  {name:<22} AUC={cv_res["test_roc_auc"].mean():.4f}±{cv_res["test_roc_auc"].std():.4f}  '
          f'F1={cv_res["test_f1"].mean():.4f}  PR-AUC={cv_res["test_pr_auc"].mean():.4f}  '
          f'Brier={-cv_res["test_brier"].mean():.4f}')

results_df = pd.DataFrame(clf_results).T.sort_values('ROC-AUC', ascending=False)
print('\n── Baseline Comparison (sorted by ROC-AUC) ──')
print(results_df.round(4))

top2_names = results_df.index[:2].tolist()
print(f'\nTop 2 models for Optuna tuning: {top2_names}')

In [ ]:
# ── ROC curves for all 4 models ────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(15, 6))
colors = ['#4C78A8', '#F28E2B', '#59A14F', '#E15759']

for ax_idx, (ax, score_type) in enumerate(zip(axes, ['ROC', 'PR'])):
    for (name, model), color in zip(clf_models.items(), colors):
        y_proba = cross_val_predict(
            model, X, y, cv=cv, method='predict_proba', n_jobs=-1
        )[:, 1]

        if score_type == 'ROC':
            fpr, tpr, _ = roc_curve(y, y_proba)
            auc = roc_auc_score(y, y_proba)
            ax.plot(fpr, tpr, lw=2, color=color, label=f'{name} (AUC={auc:.3f})')
            ax.plot([0,1],[0,1],'k--',lw=1)
            ax.set_xlabel('False Positive Rate'); ax.set_ylabel('True Positive Rate')
            ax.set_title('ROC Curves — All Models')
        else:
            prec, rec, _ = precision_recall_curve(y, y_proba)
            ap = average_precision_score(y, y_proba)
            ax.plot(rec, prec, lw=2, color=color, label=f'{name} (AP={ap:.3f})')
            ax.axhline(y.mean(), color='gray', lw=1, linestyle='--', label='Baseline')
            ax.set_xlabel('Recall'); ax.set_ylabel('Precision')
            ax.set_title('Precision-Recall Curves — All Models')

        ax.legend(loc='lower right' if score_type == 'ROC' else 'upper right', fontsize=9)
        ax.set_xlim([0, 1]); ax.set_ylim([0, 1.02])

plt.suptitle('Model Comparison — Sold Within 14 Days Classification', fontsize=12)
plt.tight_layout()
plt.show()

# ── Summary bar chart ─────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
model_names = results_df.index.tolist()
palette = sns.color_palette('Blues_d', len(model_names))

axes[0].barh(model_names, results_df['ROC-AUC'], color=palette)
axes[0].set_xlabel('ROC-AUC'); axes[0].set_title('Cross-Val AUC (higher = better)')
axes[0].set_xlim(0.45, min(1.0, results_df['ROC-AUC'].max() + 0.1))
for i, v in enumerate(results_df['ROC-AUC']):
    axes[0].text(v + 0.002, i, f'{v:.4f}', va='center', fontsize=9)

axes[1].barh(model_names, results_df['Brier'], color=palette)
axes[1].set_xlabel('Brier Score'); axes[1].set_title('Brier Score (lower = better)')
for i, v in enumerate(results_df['Brier']):
    axes[1].text(v + 0.001, i, f'{v:.4f}', va='center', fontsize=9)

plt.suptitle('Baseline Model Comparison — Time-to-Sell Classification', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# ── Optuna Tuning for Top 2 Models ─────────────────────────────────────────────
N_TRIALS = 100

def cv_auc(model):
    return cross_val_score(model, X, y, cv=cv, scoring='roc_auc', n_jobs=-1).mean()

def objective_lgbm_clf(trial):
    p = {
        'n_estimators':      trial.suggest_int('n_estimators', 100, 800),
        'learning_rate':     trial.suggest_float('learning_rate', 0.005, 0.2, log=True),
        'max_depth':         trial.suggest_int('max_depth', 3, 9),
        'num_leaves':        trial.suggest_int('num_leaves', 15, 200),
        'min_child_samples': trial.suggest_int('min_child_samples', 5, 80),
        'subsample':         trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree':  trial.suggest_float('colsample_bytree', 0.4, 1.0),
        'reg_alpha':         trial.suggest_float('reg_alpha', 1e-4, 10.0, log=True),
        'reg_lambda':        trial.suggest_float('reg_lambda', 1e-4, 10.0, log=True),
    }
    m = Pipeline([('pre', make_preprocessor()),
                  ('clf', LGBMClassifier(**p, random_state=RANDOM_STATE, verbosity=-1, n_jobs=-1))])
    return cv_auc(m)

def objective_xgb_clf(trial):
    p = {
        'n_estimators':     trial.suggest_int('n_estimators', 100, 800),
        'learning_rate':    trial.suggest_float('learning_rate', 0.005, 0.2, log=True),
        'max_depth':        trial.suggest_int('max_depth', 3, 9),
        'subsample':        trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.4, 1.0),
        'reg_alpha':        trial.suggest_float('reg_alpha', 1e-4, 10.0, log=True),
        'reg_lambda':       trial.suggest_float('reg_lambda', 1e-4, 10.0, log=True),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
        'gamma':            trial.suggest_float('gamma', 0.0, 1.0),
    }
    m = Pipeline([('pre', make_preprocessor()),
                  ('clf', XGBClassifier(**p, random_state=RANDOM_STATE, verbosity=0,
                                        n_jobs=-1, eval_metric='logloss'))])
    return cv_auc(m)

def objective_rf_clf(trial):
    p = {
        'n_estimators':      trial.suggest_int('n_estimators', 100, 600),
        'max_depth':         trial.suggest_int('max_depth', 5, 25),
        'min_samples_split': trial.suggest_int('min_samples_split', 2, 20),
        'min_samples_leaf':  trial.suggest_int('min_samples_leaf', 1, 15),
        'max_features':      trial.suggest_float('max_features', 0.3, 1.0),
    }
    m = Pipeline([('pre', make_preprocessor()),
                  ('clf', RandomForestClassifier(**p, random_state=RANDOM_STATE, n_jobs=-1))])
    return cv_auc(m)

def objective_lr_clf(trial):
    c = trial.suggest_float('C', 0.001, 10.0, log=True)
    m = Pipeline([('pre', make_preprocessor()),
                  ('clf', LogisticRegression(C=c, max_iter=1000, random_state=RANDOM_STATE, n_jobs=-1))])
    return cv_auc(m)

OBJECTIVE_MAP = {
    'LightGBM':           objective_lgbm_clf,
    'XGBoost':            objective_xgb_clf,
    'Random Forest':      objective_rf_clf,
    'Logistic Regression': objective_lr_clf,
}

print(f'Tuning top 2 models: {top2_names}  ({N_TRIALS} trials each)\n')
clf_studies = {}

for name in top2_names:
    print(f'  Tuning {name}...')
    study = optuna.create_study(
        direction='maximize', sampler=TPESampler(seed=RANDOM_STATE),
        study_name=f'sale_{name}'
    )
    study.optimize(OBJECTIVE_MAP[name], n_trials=N_TRIALS, show_progress_bar=True)
    clf_studies[name] = study
    print(f'  → Best AUC: {study.best_value:.4f}  Params: {study.best_params}\n')

print('Optuna tuning complete.')

In [ ]:
# ── Optuna visualizations ─────────────────────────────────────────────────────
from optuna.visualization.matplotlib import plot_optimization_history, plot_param_importances

for name, study in clf_studies.items():
    fig, axes = plt.subplots(1, 2, figsize=(16, 4))
    plt.sca(axes[0]); plot_optimization_history(study, target_name='ROC-AUC')
    axes[0].set_title(f'{name}: Optimization History')
    plt.sca(axes[1]); plot_param_importances(study)
    axes[1].set_title(f'{name}: Hyperparameter Importance')
    plt.tight_layout()
    plt.show()

In [ ]:
# ── Rebuild tuned models & final evaluation ────────────────────────────────────
CLF_BUILDERS = {
    'LightGBM':           lambda p: LGBMClassifier(**p, random_state=RANDOM_STATE, verbosity=-1, n_jobs=-1),
    'XGBoost':            lambda p: XGBClassifier(**p, random_state=RANDOM_STATE, verbosity=0,
                                                    n_jobs=-1, eval_metric='logloss'),
    'Random Forest':      lambda p: RandomForestClassifier(**p, random_state=RANDOM_STATE, n_jobs=-1),
    'Logistic Regression': lambda p: LogisticRegression(**p, max_iter=1000, random_state=RANDOM_STATE, n_jobs=-1),
}

tuned_clf_models = {}
tuned_auc = {}

for name in top2_names:
    best_params = clf_studies[name].best_params
    clf = CLF_BUILDERS[name](best_params)
    model = Pipeline([('pre', make_preprocessor()), ('clf', clf)])

    auc = cross_val_score(model, X, y, cv=cv, scoring='roc_auc', n_jobs=-1).mean()
    tuned_auc[name] = auc

    model.fit(X, y)
    tuned_clf_models[name] = model
    print(f'{name} — Baseline AUC: {clf_results[name]["ROC-AUC"]:.4f} → Tuned AUC: {auc:.4f}')

model_rank = sorted(tuned_clf_models.keys(), key=lambda n: tuned_auc[n], reverse=True)
best_clf_name   = model_rank[0]
second_clf_name = model_rank[1]

print(f'\n#1: {best_clf_name}  (AUC {tuned_auc[best_clf_name]:.4f})')
print(f'#2: {second_clf_name} (AUC {tuned_auc[second_clf_name]:.4f})')

In [ ]:
# ── Full classification report (OOF predictions, best model) ─────────────────
best_params = clf_studies[best_clf_name].best_params
clf_for_oof = CLF_BUILDERS[best_clf_name](best_params)
oof_model   = Pipeline([('pre', make_preprocessor()), ('clf', clf_for_oof)])

y_proba_oof = cross_val_predict(oof_model, X, y, cv=cv, method='predict_proba', n_jobs=-1)[:, 1]
y_pred_oof  = (y_proba_oof >= 0.5).astype(int)

print(f'── {best_clf_name} — Full OOF Classification Report ──')
print(classification_report(y, y_pred_oof, target_names=['Not sold 14d', 'Sold 14d']))
print(f'Overall ROC-AUC: {roc_auc_score(y, y_proba_oof):.4f}')
print(f'Brier Score:     {brier_score_loss(y, y_proba_oof):.4f}')

In [ ]:
# ── SHAP Feature Importance (best classification model) ───────────────────────
best_clf_model = tuned_clf_models[best_clf_name]
pre_fitted     = best_clf_model.named_steps['pre']
estimator      = best_clf_model.named_steps['clf']
X_transformed  = pre_fitted.transform(X)

try:
    feature_names_clean = [n.split('__')[-1] for n in pre_fitted.get_feature_names_out()]
except Exception:
    feature_names_clean = [f'f{i}' for i in range(X_transformed.shape[1])]

if best_clf_name in ('XGBoost', 'LightGBM', 'Random Forest'):
    explainer   = shap.TreeExplainer(estimator)
    shap_values = explainer.shap_values(X_transformed)
    # LightGBM may return list for binary; take class 1
    if isinstance(shap_values, list):
        shap_values = shap_values[1]
else:
    explainer   = shap.LinearExplainer(estimator, X_transformed)
    shap_values = explainer.shap_values(X_transformed)

shap_exp = shap.Explanation(
    values=shap_values,
    base_values=(explainer.expected_value[1]
                 if hasattr(explainer, 'expected_value') and isinstance(explainer.expected_value, (list, np.ndarray))
                 else (explainer.expected_value if hasattr(explainer, 'expected_value') else 0)),
    data=X_transformed,
    feature_names=feature_names_clean
)

plt.figure(figsize=(11, 8))
shap.plots.beeswarm(shap_exp, max_display=20, show=False)
plt.title(f'{best_clf_name}: SHAP Feature Importance — Sale Within 14 Days', fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
# ── Save best 2 classification models ─────────────────────────────────────────
joblib.dump(tuned_clf_models[model_rank[0]], MODELS_DIR / 'sale_model_best1.pkl')
joblib.dump(tuned_clf_models[model_rank[1]], MODELS_DIR / 'sale_model_best2.pkl')

sale_meta = {
    'feature_cols':        ALL_FEATURES,
    'numeric_features':    NUMERIC_FEATURES,
    'categorical_ordinal': CATEGORICAL_ORDINAL,
    'categorical_nominal': CATEGORICAL_NOMINAL,
    'binary_features':     BINARY_FEATURES,
    'target':              TARGET,
    'best_model_name':     model_rank[0],
    'second_model_name':   model_rank[1],
    'tuned_auc_best':      tuned_auc[model_rank[0]],
    'tuned_auc_second':    tuned_auc[model_rank[1]],
    'model_number_order':  model_number_order[0],
    'trained_at':          datetime.now().isoformat(),
}
with open(MODELS_DIR / 'sale_model_meta.json', 'w') as f:
    json.dump(sale_meta, f, indent=2)

print(f'Saved:')
print(f'  saved_models/sale_model_best1.pkl  → {model_rank[0]}  (AUC {tuned_auc[model_rank[0]]:.4f})')
print(f'  saved_models/sale_model_best2.pkl  → {model_rank[1]}  (AUC {tuned_auc[model_rank[1]]:.4f})')
print(f'  saved_models/sale_model_meta.json')

---
# Part B — Survival Analysis: Time-to-Event Modeling

Survival analysis correctly handles **right-censored** observations — listings still active at the last scrape date. The two models fit the **exact number of days** a listing stays on the market.

- **Cox PH** (semi-parametric): estimates hazard ratios; assumes proportional hazards
- **Weibull AFT** (parametric): models the log of survival time as a linear function of features

In [ ]:
from lifelines import KaplanMeierFitter, CoxPHFitter, WeibullAFTFitter
from lifelines.statistics import logrank_test

df_surv = pd.read_parquet(SURVIVAL_DATA_PATH)
print(f'Survival dataset shape: {df_surv.shape}')
print(f'Event balance (1=sold/removed):')
print(df_surv['event_sold_or_removed'].value_counts(normalize=True).map(lambda x: f'{x:.1%}'))
print(f'Duration stats:\n{df_surv["duration_days"].describe().round(2)}')

In [ ]:
# ── Kaplan-Meier overall survival curve ───────────────────────────────────────
kmf = KaplanMeierFitter()
kmf.fit(
    durations=df_surv['duration_days'],
    event_observed=df_surv['event_sold_or_removed'],
    label='All listings'
)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
kmf.plot_survival_function(ax=axes[0], ci_show=True, color='#4C78A8')
axes[0].set_title('Kaplan-Meier Survival Curve — All Listings')
axes[0].set_xlabel('Days on Market'); axes[0].set_ylabel('Probability of Still Being Listed')

# Duration histogram
sold_dur = df_surv.loc[df_surv['event_sold_or_removed']==1, 'duration_days']
censored_dur = df_surv.loc[df_surv['event_sold_or_removed']==0, 'duration_days']
axes[1].hist(sold_dur, bins=22, alpha=0.7, color='#E15759', label=f'Sold/Removed (n={len(sold_dur)})')
axes[1].hist(censored_dur, bins=22, alpha=0.7, color='#4C78A8', label=f'Still Listed (n={len(censored_dur)})')
axes[1].set_xlabel('Days on Market'); axes[1].set_ylabel('Count')
axes[1].set_title('Duration Distribution by Event Status')
axes[1].legend()

plt.suptitle('Survival Analysis Overview', fontsize=12)
plt.tight_layout()
plt.show()

print(f'Median listing survival: {kmf.median_survival_time_} days')
print(f'KM summary at day 7:  P(still listed) = {kmf.survival_function_at_times([7]).values[0]:.3f}')
print(f'KM summary at day 14: P(still listed) = {kmf.survival_function_at_times([14]).values[0]:.3f}')

In [ ]:
# ── KM curves by segment ─────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

segments = [
    ('model_number_v2', 'Model Number'),
    ('variant',         'Body Variant'),
    ('seller_type',     'Seller Type'),
]

palette = sns.color_palette('tab10', 10)

for ax, (col, label) in zip(axes, segments):
    if col not in df_surv.columns:
        ax.set_visible(False)
        continue
    groups = df_surv[col].dropna().unique()
    for i, grp in enumerate(sorted(groups)):
        mask = df_surv[col].eq(grp) & df_surv[col].notna()
        if mask.sum() < 20:
            continue
        kmf_grp = KaplanMeierFitter()
        kmf_grp.fit(
            df_surv.loc[mask, 'duration_days'],
            event_observed=df_surv.loc[mask, 'event_sold_or_removed'],
            label=str(grp)
        )
        kmf_grp.plot_survival_function(ax=ax, ci_show=False, color=palette[i % len(palette)])
    ax.set_title(f'KM Curves by {label}')
    ax.set_xlabel('Days on Market')
    ax.set_ylabel('Survival Probability')
    ax.legend(fontsize=8, loc='lower left')

plt.suptitle('Kaplan-Meier Survival Curves by Segment', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# ── Prepare survival features (numeric only for Cox/Weibull) ─────────────────
SURVIVAL_NUM_FEATURES = [
    'price_eur', 'mileage_km', 'vehicle_age_months', 'power_kw',
    'electric_range_km', 'seller_rating_stars', 'seller_rating_count',
    'image_count', 'previous_owner_count', 'warranty_months',
    'is_sportback', 'is_quattro', 'is_s_line',
    'has_matrix', 'has_pano', 'has_ahk', 'has_hud', 'has_acc', 'has_camera',
    'duplicate_listing_id', 'has_full_service_history_clean', 'had_accident_clean',
]
SURVIVAL_NUM_FEATURES = [c for c in SURVIVAL_NUM_FEATURES if c in df_surv.columns]

surv_cols = SURVIVAL_NUM_FEATURES + ['duration_days', 'event_sold_or_removed']
df_cox = df_surv[surv_cols].copy()

# Lifelines requires no NaN — impute with median
for col in SURVIVAL_NUM_FEATURES:
    df_cox[col] = pd.to_numeric(df_cox[col], errors='coerce')
    df_cox[col] = df_cox[col].fillna(df_cox[col].median())

# Convert boolean columns to float
for col in df_cox.columns:
    if df_cox[col].dtype == 'bool' or df_cox[col].dtype == 'boolean':
        df_cox[col] = df_cox[col].astype(float)

print(f'Cox/Weibull dataset: {df_cox.shape}  |  Missing: {df_cox.isna().sum().sum()}')

In [ ]:
# ── Cox Proportional Hazards ──────────────────────────────────────────────────
print('Fitting Cox Proportional Hazards model...')
cph = CoxPHFitter(penalizer=0.1)
cph.fit(df_cox, duration_col='duration_days', event_col='event_sold_or_removed')

print(f'\nCox PH C-index (concordance): {cph.concordance_index_:.4f}')
print(f'(>0.5 = better than random; 1.0 = perfect rank ordering)')
cph.print_summary(decimals=3)

In [ ]:
# ── Cox hazard ratios chart ────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 7))
cph.plot(ax=ax)
ax.set_title('Cox PH — Hazard Ratios (95% CI)\n(HR>1: faster removal; HR<1: slower removal)')
ax.axvline(0, color='gray', lw=1, linestyle='--')
plt.tight_layout()
plt.show()

In [ ]:
# ── Weibull AFT ───────────────────────────────────────────────────────────────
print('Fitting Weibull AFT model...')
waf = WeibullAFTFitter(penalizer=0.1)
waf.fit(df_cox, duration_col='duration_days', event_col='event_sold_or_removed')

print(f'\nWeibull AFT C-index: {waf.concordance_index_:.4f}')
waf.print_summary(decimals=3)

fig, ax = plt.subplots(figsize=(10, 7))
waf.plot(ax=ax)
ax.set_title('Weibull AFT — Log-Time Coefficients (95% CI)\n(coef>0: longer time on market; coef<0: faster sale)')
ax.axvline(0, color='gray', lw=1, linestyle='--')
plt.tight_layout()
plt.show()

In [ ]:
# ── Survival model comparison ──────────────────────────────────────────────────
print('── Survival Model Comparison ────────────────────────')
print(f'Cox PH C-index:      {cph.concordance_index_:.4f}')
print(f'Weibull AFT C-index: {waf.concordance_index_:.4f}')

# Predicted median survival time for 10 sample listings
sample = df_cox.head(10).drop(columns=['duration_days', 'event_sold_or_removed'])
print('\nSample predicted median time-to-removal (Weibull AFT):')
pred_median = waf.predict_median(sample)
print(pred_median.round(1).to_string())

# KM curves: price quartiles
if 'price_eur' in df_surv.columns:
    df_surv_plot = df_surv.copy()
    df_surv_plot['price_quartile'] = pd.qcut(df_surv_plot['price_eur'], q=4, labels=['Q1 Cheap','Q2','Q3','Q4 Expensive'])

    fig, ax = plt.subplots(figsize=(10, 5))
    for grp, color in zip(['Q1 Cheap','Q2','Q3','Q4 Expensive'], ['#59A14F','#4C78A8','#F28E2B','#E15759']):
        mask = df_surv_plot['price_quartile'] == grp
        kmf_q = KaplanMeierFitter()
        kmf_q.fit(
            df_surv_plot.loc[mask, 'duration_days'],
            event_observed=df_surv_plot.loc[mask, 'event_sold_or_removed'],
            label=grp
        )
        kmf_q.plot_survival_function(ax=ax, ci_show=False, color=color, lw=2)

    ax.set_title('KM Survival Curves by Price Quartile\n(lower curve = removed from market faster)')
    ax.set_xlabel('Days on Market'); ax.set_ylabel('Survival Probability')
    ax.legend()
    plt.tight_layout()
    plt.show()

print('\n── Part B complete ─────────────────────────────────────')
print('Classification models saved. Run 03_deal_finder.ipynb to score daily listings.')